# Fine-Tuning do Mistral 7B com Early Stopping

**Tech Challenge Fase 3 - FIAP Pós-Graduação em IA para Desenvolvedores**

Este notebook realiza o fine-tuning do Mistral 7B Instruct v0.3 com **early stopping**,
seguindo a metodologia padrão para encontrar o ponto ótimo de treinamento:

- **Treino único do zero** (sem retomada de checkpoints de outros treinos)
- **Checkpoints a cada 250 steps** para granularidade fina
- **Early stopping com patience=3** (para automaticamente quando eval_loss piora 3x seguidas)
- **`load_best_model_at_end=True`** — ao final, carrega o melhor checkpoint automaticamente

## Motivação

Treinos anteriores com 5 épocas mostraram **colapso catastrófico** entre os steps 1.300-1.350
(loss salta de 0,59 para 5,02 em 50 steps). A análise revelou que:
- O ponto ótimo está em torno de 1-1,3 épocas
- O cosine LR schedule com `num_train_epochs=3` garante decaimento suave
- Early stopping interrompe antes do colapso, economizando GPU

## Stack
- **QLoRA 4-bit** + **Unsloth** (2x mais rápido)
- **Trainer HuggingFace** com `EarlyStoppingCallback`
- **Dataset**: MedQuAD híbrido (~20k pares QA: 80% PT + 20% EN + BR)

---
## 1. Verificar Ambiente

Verificamos se as dependências estão instaladas e se há GPU disponível.

**Pré-requisitos (instalar antes de rodar):**
```bash
pip install unsloth peft accelerate bitsandbytes datasets transformers torch
```

In [ ]:
# ============================================================================
# VERIFICAR DEPENDÊNCIAS
# ============================================================================
# Usa importlib.util para verificar sem importar (evita conflitos de ordem)
# ============================================================================

import importlib.util

def check_package_installed(package_name):
    """Verifica se pacote está instalado sem importar."""
    spec = importlib.util.find_spec(package_name)
    return spec is not None

packages = ['unsloth', 'torch', 'torchvision', 'transformers', 'datasets', 'peft', 'accelerate', 'bitsandbytes']
missing = [p for p in packages if not check_package_installed(p)]

if missing:
    print(f"✗ Pacotes faltando: {', '.join(missing)}")
    print("")
    print("Instale com:")
    print(f"  pip install {' '.join(missing)}")
else:
    print("✓ Todas as dependências instaladas!")

In [ ]:
# ============================================================================
# IMPORTAR UNSLOTH PRIMEIRO (OBRIGATÓRIO)
# ============================================================================
# Unsloth deve ser importado antes de transformers, peft, trl para aplicar
# todas as otimizações corretamente.
# ============================================================================

import unsloth  # DEVE ser a primeira importação!

import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ GPU disponível: {gpu_name}")
    print(f"  Memória total: {gpu_memory:.1f} GB")
    
    if gpu_memory < 15:
        print("")
        print("⚠️  AVISO: GPU com menos de 16GB de VRAM.")
        print("  Pode ser necessário reduzir batch_size ou max_seq_length.")
elif torch.backends.mps.is_available():
    print("✓ Apple Silicon (MPS) disponível")
    print("  Nota: Fine-tuning em MPS pode ser mais lento que CUDA")
else:
    print("✗ ERRO: Nenhuma GPU detectada!")
    print("  Fine-tuning requer GPU com 16GB+ de VRAM.")

---
## 2. Configurar Paths Locais

Configuramos os caminhos relativos ao diretório do projeto.

In [ ]:
# ============================================================================
# CONFIGURAÇÃO DE PATHS LOCAIS
# ============================================================================

import os
from pathlib import Path

# Detectar diretório do projeto (assume que notebook está em notebooks/)
NOTEBOOK_DIR = Path(os.getcwd())
if NOTEBOOK_DIR.name == 'notebooks':
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR / "virtual-medical-assistant"

# Paths dos dados
DATA_DIR = PROJECT_DIR / "data" / "processed"
TRAINING_DATA_PATH = DATA_DIR / "training_data.jsonl"

# Paths dos modelos e checkpoints
MODELS_DIR = PROJECT_DIR / "models"
CHECKPOINT_DIR = MODELS_DIR / "checkpoints-es"  # separado dos checkpoints anteriores
FINAL_MODEL_DIR = MODELS_DIR / "medical-assistant-early-stopping"

# Criar diretórios se não existirem
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Configuração de paths:")
print(f"  Projeto: {PROJECT_DIR}")
print(f"  Dataset: {TRAINING_DATA_PATH}")
print(f"  Checkpoints: {CHECKPOINT_DIR}")
print(f"  Modelo final: {FINAL_MODEL_DIR}")

In [ ]:
# ============================================================================
# VERIFICAR SE O DATASET EXISTE
# ============================================================================

if TRAINING_DATA_PATH.exists():
    file_size = TRAINING_DATA_PATH.stat().st_size / (1024 * 1024)
    print(f"✓ Dataset encontrado: {TRAINING_DATA_PATH}")
    print(f"  Tamanho: {file_size:.2f} MB")
else:
    print(f"✗ ERRO: Dataset não encontrado!")
    print(f"  Esperado em: {TRAINING_DATA_PATH}")
    print("")
    print("  Para resolver, execute:")
    print("    python scripts/03_preparar_dataset.py")

---
## 3. Carregar e Preparar Dataset

O dataset está no formato Alpaca JSONL:
```json
{"instruction": "...", "input": "pergunta", "output": "resposta"}
```

In [ ]:
# ============================================================================
# CARREGAR DATASET
# ============================================================================

from datasets import load_dataset

# Carregar do arquivo JSONL
dataset = load_dataset('json', data_files=str(TRAINING_DATA_PATH), split='train')

print(f"✓ Dataset carregado: {len(dataset)} registros")
print(f"  Colunas: {dataset.column_names}")
print("")
print("Exemplo de registro:")
print(f"  instruction: {dataset[0]['instruction'][:80]}...")
print(f"  input: {dataset[0]['input'][:80]}...")
print(f"  output: {dataset[0]['output'][:80]}...")

In [ ]:
# ============================================================================
# DIVIDIR EM TREINO E VALIDAÇÃO
# ============================================================================

dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset_split['train']
eval_dataset = dataset_split['test']

print(f"✓ Dataset dividido:")
print(f"  Treino: {len(train_dataset)} registros (90%)")
print(f"  Validação: {len(eval_dataset)} registros (10%)")

In [ ]:
# ============================================================================
# ESTATÍSTICAS DO DATASET
# ============================================================================

import numpy as np

input_lengths = [len(x['input']) for x in dataset]
output_lengths = [len(x['output']) for x in dataset]
total_lengths = [len(x['input']) + len(x['output']) for x in dataset]

print("Estatísticas de comprimento (caracteres):")
print("")
print("  Perguntas (input):")
print(f"    Média: {np.mean(input_lengths):.0f}")
print(f"    Máximo: {np.max(input_lengths)}")
print(f"    Percentil 95: {np.percentile(input_lengths, 95):.0f}")
print("")
print("  Respostas (output):")
print(f"    Média: {np.mean(output_lengths):.0f}")
print(f"    Máximo: {np.max(output_lengths)}")
print(f"    Percentil 95: {np.percentile(output_lengths, 95):.0f}")

---
## 4. Carregar Modelo Base (Mistral 7B)

Usamos o **Unsloth** para carregar o modelo com:
- Quantização 4-bit (reduz de ~14GB para ~4GB de VRAM)
- Otimizações de velocidade (2x mais rápido)

In [ ]:
# ============================================================================
# CONFIGURAÇÕES DO MODELO
# ============================================================================

# Modelo base
MODEL_NAME = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"

# Tamanho máximo de sequência (em tokens)
MAX_SEQ_LENGTH = 2048

# Configuração de precisão
DTYPE = None  # Auto-detectar
LOAD_IN_4BIT = True  # Quantização 4-bit

print("Configuração do modelo:")
print(f"  Modelo: {MODEL_NAME}")
print(f"  Max seq length: {MAX_SEQ_LENGTH}")
print(f"  Quantização: 4-bit")

In [ ]:
# ============================================================================
# CARREGAR MODELO E TOKENIZER
# ============================================================================

from unsloth import FastLanguageModel

print("Carregando modelo (pode demorar alguns minutos)...")
print("")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print("")
print("✓ Modelo carregado com sucesso!")

---
## 5. Configurar LoRA

**LoRA (Low-Rank Adaptation)** permite treinar apenas uma pequena fração dos parâmetros.

In [ ]:
# ============================================================================
# CONFIGURAÇÃO DO LORA
# ============================================================================

model = FastLanguageModel.get_peft_model(
    model,
    r=64,                    # Rank do LoRA
    lora_alpha=128,          # Scaling factor
    lora_dropout=0.05,       # Dropout para regularização
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
percent = 100 * trainable_params / total_params

print(f"\n✓ LoRA configurado!")
print(f"  Parâmetros treináveis: {trainable_params:,} ({percent:.2f}%)")
print(f"  Parâmetros totais: {total_params:,}")

---
## 6. Preparar Formato de Treinamento

In [ ]:
# ============================================================================
# TEMPLATE ALPACA EM PORTUGUÊS
# ============================================================================

ALPACA_PROMPT = """Abaixo está uma instrução que descreve uma tarefa, junto com uma entrada que fornece contexto adicional. Escreva uma resposta que complete adequadamente a solicitação.

### Instrução:
{}

### Entrada:
{}

### Resposta:
{}"""

EOS_TOKEN = tokenizer.eos_token

print("Template Alpaca configurado.")

In [ ]:
# ============================================================================
# FUNÇÃO DE FORMATAÇÃO DO DATASET
# ============================================================================

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]

    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = ALPACA_PROMPT.format(instruction, input_text, output) + EOS_TOKEN
        texts.append(text)

    return {"text": texts}

train_dataset_formatted = train_dataset.map(
    formatting_prompts_func,
    batched=True,
    desc="Formatando treino"
)

eval_dataset_formatted = eval_dataset.map(
    formatting_prompts_func,
    batched=True,
    desc="Formatando validação"
)

print(f"\n✓ Dataset formatado!")
print(f"  Treino: {len(train_dataset_formatted)} exemplos")
print(f"  Validação: {len(eval_dataset_formatted)} exemplos")

---
## 7. Configurar Treinamento com Early Stopping

Diferenças em relação ao treino anterior:
- **3 épocas** (máximo) — early stopping para antes se necessário
- **Sem `max_steps`** — o cosine LR schedule decai naturalmente até zero
- **`eval_steps=250`** — avaliação mais frequente para detectar degradação cedo
- **`save_steps=250`** — checkpoints granulares
- **`EarlyStoppingCallback(patience=3)`** — para após 3 avaliações sem melhora

In [ ]:
# ============================================================================
# CONFIGURAÇÃO DO TREINAMENTO COM EARLY STOPPING
# ============================================================================

from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from transformers import EarlyStoppingCallback

# Detectar capacidade da GPU para ajustar batch size
if torch.cuda.is_available():
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_memory >= 24:  # A100, RTX 4090, etc.
        BATCH_SIZE = 4
        GRAD_ACCUM = 4
    elif gpu_memory >= 16:  # RTX 3090, A10, etc.
        BATCH_SIZE = 2
        GRAD_ACCUM = 8
    else:  # GPUs menores
        BATCH_SIZE = 1
        GRAD_ACCUM = 16
else:
    BATCH_SIZE = 1
    GRAD_ACCUM = 16

# ~1.156 steps/epoch → 3 epochs ≈ 3.468 steps
# Early stopping interrompe antes se eval_loss piora 3x seguidas
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),

    # ===== Epochs e Batching =====
    num_train_epochs=3,                # Máximo 3 — early stopping para antes
    # SEM max_steps — cosine LR decai até zero naturalmente
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    # ===== Learning Rate =====
    learning_rate=2e-4,
    warmup_steps=200,
    weight_decay=0.01,
    lr_scheduler_type="cosine",

    # ===== Precisão =====
    fp16=torch.cuda.is_available(),

    # ===== Checkpoints (granulares) =====
    save_strategy="steps",
    save_steps=250,
    save_total_limit=15,               # Manter mais checkpoints para análise

    # ===== Avaliação (frequente) =====
    eval_strategy="steps",
    eval_steps=250,

    # ===== Early Stopping =====
    load_best_model_at_end=True,       # Carrega melhor checkpoint ao final
    metric_for_best_model="eval_loss", # Critério: menor eval_loss
    greater_is_better=False,           # Menor = melhor para loss

    # ===== Logging =====
    logging_steps=50,
    report_to="none",

    # ===== Otimizações =====
    optim="adamw_8bit",
    seed=42,
    data_seed=42,                      # Seed explícita para ordem dos dados
)

# Steps estimados
steps_per_epoch = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
total_steps = steps_per_epoch * 3
print("✓ TrainingArguments configurado!")
print(f"  Epochs (máximo): 3")
print(f"  Steps/epoch: ~{steps_per_epoch}")
print(f"  Steps máximos: ~{total_steps}")
print(f"  Batch efetivo: {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Checkpoints: a cada 250 steps")
print(f"  Avaliação: a cada 250 steps")
print(f"  Early stopping: patience=3 (para após 750 steps sem melhora)")
print(f"  Seed: 42 (dados e treino)")

In [ ]:
# ============================================================================
# CRIAR TRAINER COM EARLY STOPPING
# ============================================================================

def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("Tokenizando datasets...")
train_tokenized = train_dataset_formatted.map(
    tokenize_function, 
    batched=True, 
    remove_columns=["text", "instruction", "input", "output"],
    desc="Tokenizando treino"
)
eval_tokenized = eval_dataset_formatted.map(
    tokenize_function, 
    batched=True, 
    remove_columns=["text", "instruction", "input", "output"],
    desc="Tokenizando validação"
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, 
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    data_collator=data_collator,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3),
    ],
)

print("✓ Trainer criado com EarlyStoppingCallback(patience=3)")

---
## 8. Treinar!

O treino começa **do zero** (baseline Mistral 7B) — sem retomada de checkpoints anteriores.

Early stopping monitora `eval_loss` a cada 250 steps. Se piora 3 avaliações seguidas, o treino é interrompido e o melhor checkpoint é carregado automaticamente.

**Tempo estimado** (antes do early stopping parar, ~1.000-2.000 steps):
- RTX 4090 (24GB): ~1-2 horas
- RTX 3090 (24GB): ~2-3 horas
- A100 (40GB): ~1-1.5 horas

In [ ]:
# ============================================================================
# TREINAR O MODELO (DO ZERO)
# ============================================================================

print("=" * 60)
print("  INICIANDO TREINAMENTO COM EARLY STOPPING")
print("=" * 60)
print("")
print("Configuração:")
print(f"  Epochs máximos: 3")
print(f"  Early stopping patience: 3 (avaliações a cada 250 steps)")
print(f"  O treino será interrompido automaticamente quando eval_loss")
print(f"  parar de melhorar por 3 avaliações consecutivas.")
print("")

trainer_stats = trainer.train()

print("")
print("=" * 60)
if trainer.state.global_step < total_steps:
    print(f"  EARLY STOPPING ATIVADO no step {trainer.state.global_step}!")
else:
    print("  TREINAMENTO CONCLUÍDO (todas as épocas)")
print("=" * 60)
print(f"  Melhor checkpoint: {trainer.state.best_model_checkpoint}")
print(f"  Melhor eval_loss: {trainer.state.best_metric:.6f}")

In [ ]:
# ============================================================================
# ESTATÍSTICAS DO TREINAMENTO
# ============================================================================

hours = trainer_stats.metrics['train_runtime'] / 3600
print("Estatísticas do treinamento:")
print(f"  Tempo total: {hours:.2f} horas")
print(f"  Steps executados: {trainer_stats.global_step}")
print(f"  Loss final (treino): {trainer_stats.metrics['train_loss']:.4f}")
print(f"  Samples/segundo: {trainer_stats.metrics['train_samples_per_second']:.2f}")
print("")
print(f"  Melhor eval_loss: {trainer.state.best_metric:.6f}")
print(f"  Melhor checkpoint: {trainer.state.best_model_checkpoint}")
print(f"  Step do melhor: {trainer.state.best_global_step}")

# Mostrar evolução do eval_loss
print("")
print("Evolução do eval_loss:")
print(f"  {'Step':>6}  {'Eval Loss':>10}  {'Status'}")
print(f"  {'─'*6}  {'─'*10}  {'─'*20}")
best_loss = float('inf')
for entry in trainer.state.log_history:
    if 'eval_loss' in entry:
        step = entry['step']
        loss = entry['eval_loss']
        marker = ""
        if loss < best_loss:
            best_loss = loss
            marker = "← melhor"
        elif step == trainer.state.best_global_step:
            marker = "← melhor (carregado)"
        print(f"  {step:>6}  {loss:>10.6f}  {marker}")

---
## 9. Salvar Modelo Final

O Trainer já carregou o melhor checkpoint automaticamente (`load_best_model_at_end=True`).
Agora salvamos os adapters LoRA e o tokenizer para uso posterior.

In [ ]:
# ============================================================================
# SALVAR MODELO FINAL
# ============================================================================

print(f"Salvando modelo em: {FINAL_MODEL_DIR}")
print("")

model.save_pretrained(str(FINAL_MODEL_DIR))
print("✓ Adapters LoRA salvos")

tokenizer.save_pretrained(str(FINAL_MODEL_DIR))
print("✓ Tokenizer salvo")

print("")
print("Arquivos salvos:")
total_size = 0
for f in FINAL_MODEL_DIR.iterdir():
    size = f.stat().st_size / (1024 * 1024)
    total_size += size
    print(f"  {f.name}: {size:.2f} MB")

print(f"")
print(f"Tamanho total: {total_size:.2f} MB")

---
## 9.1. Upload do Modelo para o Hugging Face Hub

O modelo fine-tuned (adapters LoRA ~640MB) é grande demais para o GitHub (limite 100MB).
Usamos o **Hugging Face Hub** como repositório do modelo.

**Pré-requisitos:**
1. Criar conta em [huggingface.co](https://huggingface.co)
2. Criar token de acesso em [Settings > Tokens](https://huggingface.co/settings/tokens) com permissão **Write**
3. Configurar no `.env` do projeto:
   ```
   HF_TOKEN=hf_xxxxxxxxxxxxxxxxxxxx
   HF_HUB_REPO=seu-usuario/medical-assistant-mistral-7b-ft
   ```

In [ ]:
# ============================================================================
# UPLOAD DO MODELO PARA O HUGGING FACE HUB
# ============================================================================

import os
from pathlib import Path
from huggingface_hub import HfApi, login

# Tentar carregar .env do projeto
env_path = PROJECT_DIR / ".env"
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, _, value = line.partition("=")
            os.environ.setdefault(key.strip(), value.strip())

# --- CONFIGURAÇÃO ---
HF_TOKEN = os.getenv("HF_TOKEN", "")
HF_HUB_REPO = os.getenv("HF_HUB_REPO", "zagari/medical-assistant-mistral-7b-ft")

if not HF_TOKEN:
    print("✗ ERRO: HF_TOKEN não configurado.")
    print("  Opção 1: Preencha no arquivo .env do projeto")
    print("  Opção 2: Cole o token diretamente na variável HF_TOKEN acima")
    print("")
    print("  Crie seu token em: https://huggingface.co/settings/tokens (permissão Write)")
else:
    login(token=HF_TOKEN)
    print(f"✓ Autenticado no Hugging Face Hub")

    api = HfApi()
    api.create_repo(repo_id=HF_HUB_REPO, exist_ok=True, private=False)

    best_step = trainer.state.best_global_step
    commit_msg = (
        f"Upload adapters LoRA fine-tuned com early stopping "
        f"(melhor checkpoint: step {best_step}, eval_loss: {trainer.state.best_metric:.6f})"
    )

    print(f"Enviando modelo para: https://huggingface.co/{HF_HUB_REPO}")
    print(f"  {commit_msg}")
    print("")

    api.upload_folder(
        folder_path=str(FINAL_MODEL_DIR),
        repo_id=HF_HUB_REPO,
        commit_message=commit_msg,
    )

    print(f"✓ Modelo enviado com sucesso!")
    print(f"  URL: https://huggingface.co/{HF_HUB_REPO}")
    print(f"  Melhor step: {best_step}")
    print("")
    print("Para usar na aplicação, configure no .env:")
    print(f"  FINE_TUNED_MODEL={HF_HUB_REPO}")

---
## 10. Testar Inferência

O modelo carregado é o **melhor checkpoint** selecionado pelo early stopping.

In [ ]:
# ============================================================================
# PREPARAR MODELO PARA INFERÊNCIA
# ============================================================================

FastLanguageModel.for_inference(model)
print("✓ Modelo preparado para inferência")

In [ ]:
# ============================================================================
# FUNÇÃO DE INFERÊNCIA
# ============================================================================

def ask_medical_question(question: str, max_new_tokens: int = 512) -> str:
    prompt = ALPACA_PROMPT.format(
        "Responda a seguinte pergunta médica de forma clara e detalhada.",
        question,
        ""
    )

    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    inputs = tokenizer([prompt], return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=0.7,
        top_p=0.9,
    )

    response = tokenizer.batch_decode(outputs)[0]

    if "### Resposta:" in response:
        response = response.split("### Resposta:")[1].strip()
        if EOS_TOKEN in response:
            response = response.split(EOS_TOKEN)[0].strip()

    return response

In [ ]:
# ============================================================================
# TESTAR COM PERGUNTAS MÉDICAS
# ============================================================================

perguntas_teste = [
    "Quais são os sintomas do diabetes tipo 2?",
    "Como é feito o diagnóstico de hipertensão arterial?",
    "Quais são os tratamentos disponíveis para asma?",
]

print("=" * 60)
print("  TESTE DE INFERÊNCIA")
print("=" * 60)

for i, pergunta in enumerate(perguntas_teste, 1):
    print(f"\n--- Pergunta {i} ---")
    print(f"Q: {pergunta}")
    print("")

    resposta = ask_medical_question(pergunta)

    print(f"R: {resposta}")
    print("-" * 40)

---
## 11. Carregar Modelo Salvo (Para Uso Futuro)

Duas opções para carregar o modelo fine-tuned:
- **Local**: a partir da pasta `models/medical-assistant-early-stopping`
- **Hugging Face Hub**: a partir do repositório remoto

In [ ]:
# ============================================================================
# OPÇÃO A: CARREGAR MODELO LOCAL
# ============================================================================
# Usa o path definido no início do notebook (FINAL_MODEL_DIR).
# Requer que o modelo tenha sido salvo localmente na seção 10.
# ============================================================================

from unsloth import FastLanguageModel
from pathlib import Path

model_local, tokenizer_local = FastLanguageModel.from_pretrained(
    model_name=str(FINAL_MODEL_DIR),
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model_local)
print(f"✓ Modelo carregado do path local: {FINAL_MODEL_DIR}")

In [ ]:
# ============================================================================
# OPÇÃO B: CARREGAR MODELO DO HUGGING FACE HUB
# ============================================================================
# Baixa automaticamente os adapters LoRA do repositório remoto.
# Útil para quem não tem o modelo salvo localmente.
# ============================================================================

from unsloth import FastLanguageModel

HF_REPO = "zagari/medical-assistant-mistral-7b-ft"

model_hub, tokenizer_hub = FastLanguageModel.from_pretrained(
    model_name=HF_REPO,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model_hub)
print(f"✓ Modelo carregado do Hugging Face Hub: {HF_REPO}")

---
## Próximos Passos

Após o fine-tuning com early stopping:

1. **Comparar com checkpoint-1500 anterior**
   - Rodar `scripts/06_avaliar_finetuned.py` apontando para `models/medical-assistant-early-stopping`
   - Comparar métricas (BLEU, ROUGE, similaridade semântica) vs o modelo anterior

2. **Exportar GGUF** (se este modelo for o vencedor)
   - `scripts/08_exportar_gguf.py --model-path models/medical-assistant-early-stopping`

3. **Documentação**
   - Registrar curva de early stopping e step ótimo no relatório LaTeX

In [ ]:
# ============================================================================
# FIM DO NOTEBOOK
# ============================================================================

print("Fine-tuning com early stopping concluído!")
print("")
print(f"  Melhor checkpoint: step {trainer.state.best_global_step}")
print(f"  Melhor eval_loss: {trainer.state.best_metric:.6f}")
print(f"  Modelo salvo em: {FINAL_MODEL_DIR}")
print("")
print("Próximos passos:")
print("  1. Comparar com checkpoint-1500 via scripts de avaliação")
print("  2. Exportar GGUF do melhor modelo")
print("  3. Documentar resultados no relatório")